# Inspect Ai metric experiments

Every metric has a separate explanation and test cell. Shared fixtures provide 10 RAG and 10 agent examples. These cells prepare labeled inputs; actual judge scores require the selected framework evaluator and model.

In [ ]:
import json
from pathlib import Path

experiment = {'framework': 'Inspect AI', 'rag_scorers': ['retrieval_relevance', 'evidence_coverage', 'groundedness', 'answer_correctness', 'safety'], 'agent_scorers': ['task_success', 'tool_use_correctness', 'trajectory_quality', 'step_budget', 'policy_compliance'], 'task_record_fields': ['input', 'reference', 'retrieved_documents', 'response', 'tool_trace'], 'status': 'plan_only: implement task, solver, and scorers'}

def load_shared_cases(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder / 'datasets' / filename, folder / 'opensource' / 'datasets' / filename):
            if candidate.is_file():
                return json.loads(candidate.read_text(encoding='utf-8'))
    raise FileNotFoundError(f'Cannot find shared dataset: {filename}')

rag_cases = load_shared_cases('rag_cases.json')
agent_cases = load_shared_cases('agent_cases.json')
assert len(rag_cases) >= 10 and len(agent_cases) >= 10
assert all({'question','contexts','response','reference','expected_context_relevance'} <= case.keys() for case in rag_cases)
assert all({'task','tool_trace','expected_tools','expected_outcome','final_response'} <= case.keys() for case in agent_cases)
print(f'Loaded {len(rag_cases)} RAG and {len(agent_cases)} agentic AI examples.')


## RAG metrics

The metric checks below are isolated so you can inspect and run them individually.

### Retrieval Relevance

**What it measures:** Checks whether retrieved passages are useful for the current query.

**Why/when to use it:** Use it to distinguish retrieval failures from generation failures.

**Example fields:** question, contexts, expected_context_relevance.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'retrieval_relevance'
metric_fields = ['question', 'contexts', 'expected_context_relevance']
metric_label = 'expected_context_relevance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Evidence Coverage

**What it measures:** Checks whether retrieved evidence contains facts needed to support the expected answer.

**Why/when to use it:** Use it to find retrieval omissions; requires reference or required-evidence labels.

**Example fields:** contexts, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'evidence_coverage'
metric_fields = ['contexts', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Groundedness

**What it measures:** Assesses whether the response is supported by retrieved context or tool evidence.

**Why/when to use it:** Use it to detect unsupported claims in RAG answers and agent summaries.

**Example fields:** response, contexts.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'groundedness'
metric_fields = ['response', 'contexts']
metric_label = 'expected_grounded'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Answer Correctness

**What it measures:** Compares generated answer with a trusted reference, including semantic factual agreement.

**Why/when to use it:** Use it when references or expert rubrics exist; it differs from grounding because a grounded answer can still omit required facts.

**Example fields:** question, response, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_correctness'
metric_fields = ['question', 'response', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Safety

**What it measures:** Checks whether responses follow the configured safety policy.

**Why/when to use it:** Use policy-specific test cases; factuality metrics are not a substitute for safety evaluation.

**Example fields:** question, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'safety'
metric_fields = ['question', 'response']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


## Agentic AI metrics

The metric checks below are isolated so you can inspect and run them individually.

### Task Success

**What it measures:** Measures whether the agent accomplished the requested goal.

**Why/when to use it:** Use as end-to-end success; include cases where the agent falsely claims success.

**Example fields:** task, final_response, expected_outcome.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'task_success'
metric_fields = ['task', 'final_response', 'expected_outcome']
metric_label = 'expected_task_success'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Use Correctness

**What it measures:** Evaluates whether the agent chose and used suitable tools to progress the task.

**Why/when to use it:** Use it to detect ineffective or prohibited actions.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_use_correctness'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = 'expected_tool_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Trajectory Quality

**What it measures:** Evaluates whether intermediate actions are coherent and make progress toward the goal.

**Why/when to use it:** Use it to find inefficient or invalid traces even when the final answer is correct.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'trajectory_quality'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Step Budget

**What it measures:** Checks whether the agent stays within an action or call limit.

**Why/when to use it:** Use as a deterministic latency/cost guardrail and separately measure task success.

**Example fields:** tool_trace, max_steps.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'step_budget'
metric_fields = ['tool_trace', 'max_steps']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Policy Compliance

**What it measures:** Checks whether actions and answers follow operational and safety rules.

**Why/when to use it:** Use policy-specific positive and negative cases; success does not prove compliance.

**Example fields:** task, tool_trace, final_response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'policy_compliance'
metric_fields = ['task', 'tool_trace', 'final_response']
metric_label = 'expected_policy_compliance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})
